# Part 1 — SOP as a graph

Procedure compliance follows SOP order, not free text.

Given a request to start a step, return allowed / blocked from the dependency graph.

## Definition

Procedure = directed acyclic graph (DAG).

- $S = \{s_1,\ldots,s_n\}$: steps (nodes)
- $s_i \rightarrow s_j$: $s_i$ must finish before $s_j$ starts
- $\mathrm{Pred}(s)$: direct predecessors of $s$
- $C \subseteq S$: completed steps

**Allow rule.** $s$ is allowed iff $\mathrm{Pred}(s) \subseteq C$.

Block reasons: $\mathrm{Pred}(s) \setminus C$.

In [1]:
from dataclasses import dataclass, field


@dataclass
class Step:
    id: str
    title: str
    pred: list[str] = field(default_factory=list)  # Pred(s)
    risk: str = "low"


@dataclass
class WorkOrder:
    id: str
    completed: set[str] = field(default_factory=set)  # C


def is_allowed(step: Step, order: WorkOrder) -> tuple[bool, set[str]]:
    """Return (allowed, Pred(s) \\ C)."""
    missing = set(step.pred) - order.completed
    return len(missing) == 0, missing

## Example

Chain $s_1 \rightarrow s_2 \rightarrow s_3$.

[`docs/sop_samples/assembly_sop_v1.md`](../docs/sop_samples/assembly_sop_v1.md)

$\mathrm{Pred}(s_1)=\emptyset$, $\mathrm{Pred}(s_2)=\{s_1\}$, $\mathrm{Pred}(s_3)=\{s_2\}$.

If $C=\{s_1\}$, then $s_3$ is blocked: $\mathrm{Pred}(s_3)\setminus C = \{s_2\}$.

In [2]:
steps = {
    "STEP-01": Step("STEP-01", "Pick base plate"),
    "STEP-02": Step("STEP-02", "Align bracket", pred=["STEP-01"]),
    "STEP-03": Step("STEP-03", "Robot pick", pred=["STEP-02"], risk="high"),
}

order = WorkOrder(id="WO-100", completed={"STEP-01"})

for sid, step in steps.items():
    ok, missing = is_allowed(step, order)
    print(f"{sid}: allowed={ok}, missing={sorted(missing)}")

STEP-01: allowed=True, missing=[]
STEP-02: allowed=True, missing=[]
STEP-03: allowed=False, missing=['STEP-02']


## Trick: set difference

Block reasons are the set difference $\mathrm{Pred}(s) \setminus C$.

Only direct predecessors are stored; order along the chain covers the rest.